In [1]:
import pandas as pd
from pathlib import Path
import numpy as np
import k3d
import vtk
import opengate as gate
from scipy.spatial.transform import Rotation

In [6]:
# 1. Import the VRML scene using VTK
importer = vtk.vtkVRMLImporter()
importer.SetFileName("../validation_geometry.wrl")
importer.Update()

# 2. Access the renderer to retrieve the geometric shapes (actors)
render_window = importer.GetRenderWindow()
renderer = render_window.GetRenderers().GetFirstRenderer()
actors = renderer.GetActors()

# 3. Initialize an empty K3D plot
plot = k3d.plot(camera_auto_fit=False)

# [pos_x, pos_y, pos_z, target_x, target_y, target_z, up_x, up_y, up_z]
plot.camera = [
    300, 300, 300,  # Camera Position
    0, 0, 0,        # Looking at the origin (center of your geometry)
    0, 0, 1         # Z-axis is "up"
]

# 4. Iterate over each piece of the geometry, extract it, and add to K3D
actors.InitTraversal()
number_of_actors = actors.GetNumberOfItems()
print(f"Total number of actors in the scene: {number_of_actors}")

all_actors = [actors.GetNextActor() for i in range(number_of_actors)]

for i, actor in enumerate(all_actors):
    # if i<62 and (i == 0 or i == number_of_actors-1 or (i-1) % 3 !=2):
    #     continue  # Skip the first and last actors

    
    polydata = actor.GetMapper().GetInput()
    # --- Edge Extraction and Tubing ---
    
    # 1. Clean the mesh (weld duplicate vertices)
    cleaner = vtk.vtkCleanPolyData()
    cleaner.SetInputData(polydata)
    cleaner.Update()

    # 2. Extract the feature edges from the cleaned mesh
    edge_filter = vtk.vtkFeatureEdges()
    edge_filter.SetInputConnection(cleaner.GetOutputPort())
    edge_filter.BoundaryEdgesOn()       
    edge_filter.FeatureEdgesOn()        
    edge_filter.SetFeatureAngle(30.0)   
    edge_filter.NonManifoldEdgesOff()
    edge_filter.ManifoldEdgesOff()
    edge_filter.Update()

    # 3. Turn the 1D lines into 3D tubes for K3D rendering
    tube_filter = vtk.vtkTubeFilter()
    tube_filter.SetInputConnection(edge_filter.GetOutputPort())
    tube_filter.SetRadius(0.5) # *** ADJUST THIS RADIUS based on your geometry scale ***
    tube_filter.SetNumberOfSides(6)
    tube_filter.Update()

    # Get the thickened sharp edges
    sharp_edges_polydata = tube_filter.GetOutput()
    if sharp_edges_polydata.GetNumberOfPoints() == 0:
        print(f"Actor {i} has no sharp edges, skipping edge rendering.")
    else:
        # Add the sharp edges to the plot (in red)
        plot += k3d.vtk_poly_data(sharp_edges_polydata, color=0xFF0000)
    
    # ----------------------------------

    # Plot the original surfaces in transparent blue
    plot += k3d.vtk_poly_data(polydata, color=0x0072BD, wireframe=False, opacity=0.5)

# 5. Render the final output
plot.display()

[2026-04-07 11:58:51,953] INFO in helpers: Converting int64 array to int32 for JS compatibility.
[2026-04-07 11:58:51,955] INFO in helpers: Converting int64 array to int32 for JS compatibility.
[2026-04-07 11:58:51,965] INFO in helpers: Converting int64 array to int32 for JS compatibility.
[2026-04-07 11:58:51,967] INFO in helpers: Converting int64 array to int32 for JS compatibility.
[2026-04-07 11:58:51,970] INFO in helpers: Converting int64 array to int32 for JS compatibility.
[2026-04-07 11:58:51,972] INFO in helpers: Converting int64 array to int32 for JS compatibility.
[2026-04-07 11:58:51,975] INFO in helpers: Converting int64 array to int32 for JS compatibility.
[2026-04-07 11:58:51,977] INFO in helpers: Converting int64 array to int32 for JS compatibility.
[2026-04-07 11:58:51,979] INFO in helpers: Converting int64 array to int32 for JS compatibility.
[2026-04-07 11:58:51,981] INFO in helpers: Converting int64 array to int32 for JS compatibility.
[2026-04-07 11:58:51,984] INFO

Total number of actors in the scene: 23
Actor 22 has no sharp edges, skipping edge rendering.


Output()